In [18]:
import os
import pandas as pd
from tqdm import tqdm

In [13]:
filename_folder_reads = '/scratch/jplfaria/ReadMapping/data/'
filename_folder_output = '/scratch/shared/CDM/salterns/reads/paired'

In [4]:
path_reads = '/scratch/jplfaria/ReadMapping/data/'
path_metagenome_assembly = '/scratch/chenry/Projects/Salterns/nboutput/metagenome_assemblies/'
reads_fastq = set()
for f in os.listdir(path_reads):
    if f.endswith('.fastq'):
        reads_fastq.add(f)
found_metagenome_assembly = set()
for f in os.listdir(path_metagenome_assembly):
    if f.endswith('.fna.filtered.fa'):
        found_metagenome_assembly.add(f)

In [7]:
sample_data = pd.read_csv('/home/fliu/cliff_mags/data/sample_read_assembly.tsv', sep='\t', index_col=0).transpose().to_dict()

## Split reads

In [21]:
from modelseedpy_ext.reads.reads_proc import split_reads
for sample_id in tqdm(sample_data):
    read_id = sample_data[sample_id].get('Reads')
    assembly_id = sample_data[sample_id].get('Assembly v2 (2018)')
    if not pd.isna(read_id) and not pd.isna(assembly_id):
        read_id = read_id[:-3]
        if read_id not in reads_fastq:
            raise ValueError(f'not found: {read_id}')
        assembly_id = assembly_id + '.filtered.fa'
        if assembly_id not in found_metagenome_assembly:
            raise ValueError(f'not found: {assembly_id}')
            
        filename_fastq = f'{filename_folder_reads}/{read_id}'
        filename_out_fastq_1 = f'{filename_folder_output}/{sample_id}_1.fastq'
        filename_out_fastq_2 = f'{filename_folder_output}/{sample_id}_2.fastq'
        if os.path.exists(filename_out_fastq_2) and os.path.exists(filename_out_fastq_1):
            pass
        else:
            if os.path.exists(filename_fastq):
                print(sample_id, filename_fastq)
                split_reads(filename_fastq, filename_out_fastq_1, filename_out_fastq_2)

100%|██████████| 38/38 [00:00<00:00, 21696.64it/s]


## Eval reads

In [17]:
sample_id = 'SF2_C_D1_MG'
filename_fastq_1 = f'/scratch/shared/CDM/salterns/reads/paired/{sample_id}_1.fastq'
filename_fastq_2 = f'/scratch/shared/CDM/salterns/reads/paired/{sample_id}_2.fastq'
read_end = set()
unique_h = set()
with open(filename_fastq_1, 'r') as fh:
    line_header = fh.readline()
    while line_header:
        read_id, read_x = line_header.split(' ')
        end = int(read_x[0])
        read_end.add(end)
        line_seq = fh.readline()
        line_sep = fh.readline()
        line_qua = fh.readline()
        if read_id not in unique_h:
            unique_h.add(read_id)
        else:
            raise ValueError(f'not unique {read_id}')
        line_header = fh.readline()
print(read_end, len(unique_h))
read_end = set()
unique_h = set()
with open(filename_fastq_2, 'r') as fh:
    line_header = fh.readline()
    while line_header:
        read_id, read_x = line_header.split(' ')
        end = int(read_x[0])
        read_end.add(end)
        line_seq = fh.readline()
        line_sep = fh.readline()
        line_qua = fh.readline()
        if read_id not in unique_h:
            unique_h.add(read_id)
        else:
            raise ValueError(f'not unique {read_id}')
        line_header = fh.readline()
print(read_end, len(unique_h))

# Map reads to contigs

In [52]:
import pymongo
from cobrakbase import KBaseCache
from modelseedpy_ext.re.core.genome import REAssembly
from modelseedpy_ext.re.hash_seq import HashSeq
kbase = KBaseCache(path='/scratch/fliu/data/kbase/cache/')
mongo_client = pymongo.MongoClient('mongodb://sequoia.mcs.anl.gov:27017')
mongo_project = mongo_client['salterns']

### Index assembly/genome in database

In [65]:
genomes = {}
genome_assembly = {}
for info in tqdm(kbase.list_workspace(155805)):
    if info.type == 'KBaseGenomes.Genome':
        genome = kbase.get_from_ws(str(info))
        assembly = kbase.get_from_ws(genome.assembly_ref)
        genomes[info.id] = genome
        genome_assembly[info.id] = assembly
        #print(info.id, info.type)
        filename_fna = f'/scratch/fliu/data/kbase/cache/handle/{assembly.fasta_handle_ref}'
        if os.path.exists(filename_fna):
            contigs = REAssembly.from_fasta(filename_fna)

            h_assembly = contigs.hash_value

            doc_assembly = mongo_project['assembly'].find_one({'_id': contigs.hash_value})
            if doc_assembly is None:
                mongo_project['assembly'].insert_one({
                    '_id': h_assembly,
                    'path': filename_fna,
                    'default_kbase_genome': genome.info.to_kbase_json(),
                    'default_kbase_assembly': assembly.info.to_kbase_json(),
                    'handle_ref': assembly.fasta_handle_ref,
                    'taxonomy_gtdb': None,
                })

            for feature_contig in contigs.features:
                h_contig = HashSeq(feature_contig.seq)
                doc_contig = mongo_project['contig'].find_one({'_id': h_contig.hash_value})
                if doc_contig is None:
                    mongo_project['contig'].insert_one({
                        '_id': h_contig.hash_value,
                        'in_assembly': [],
                        'sequence': feature_contig.seq
                    })
                doc_updated = mongo_project['contig'].find_one_and_update(
                    {'_id': h_contig.hash_value},
                    {"$addToSet": {"in_assembly": h_assembly}},
                    return_document=True
                )
                if doc_updated is None:
                    raise ValueError("not found!")

100%|██████████| 1298/1298 [08:08<00:00,  2.66it/s] 


In [ ]:
from modelseedpy.core.msgenome import MSGenome, MSFeature
for doc in mongo_project['assembly'].find():
    h_contigset = doc['_id']
    filename_fna = f'/scratch/fliu/data/cliff/genomes/fna/{h_contigset}.fna'
    if not os.path.exists(filename_fna):
        contigs = REAssembly.from_fasta(doc['path'])
        features = {}
        for feature_contig in contigs.features:
            h_contig = HashSeq(feature_contig.seq)
            feature = MSFeature(h_contig.hash_value, feature_contig.seq)
            if feature.id not in features:
                features[feature.id] = feature
            else:
                print('!')
        fna = MSGenome()
        fna.add_features(list(features.values()))
        fna.to_fasta(filename_fna)

### Generate assembly library file

In [86]:
with open(f'/scratch/fliu/data/cliff/genomes/ani_library_derep_genomes.txt', 'w') as fh:
    for doc in mongo_project['assembly'].find():
        h_contigset = doc['_id']
        filename_fna = f'/scratch/fliu/data/cliff/genomes/fna/{h_contigset}.fna'
        fh.write(f'{filename_fna}\n')

### Generate CoverM run script

In [95]:
cmds = []
for sample_id in sample_data:
    read_id = sample_data[sample_id].get('Reads')
    assembly_id = sample_data[sample_id].get('Assembly v2 (2018)')
    if not pd.isna(read_id) and not pd.isna(assembly_id):
        read_id = read_id[:-3]
        if read_id not in reads_fastq:
            raise ValueError(f'not found: {read_id}')
        assembly_id = assembly_id + '.filtered.fa'
        if assembly_id not in found_metagenome_assembly:
            raise ValueError(f'not found: {assembly_id}')
        
        bam_file_cache_directory = f"/scratch/fliu/data/cliff/reads/coverm/{sample_id}/"
        bam_file_cache_directory_bins = f"/scratch/fliu/data/cliff/reads/coverm/{sample_id}/bins"
        if not os.path.exists(bam_file_cache_directory):
            os.mkdir(bam_file_cache_directory)
            print(f'crated folder: {bam_file_cache_directory}')
        if not os.path.exists(bam_file_cache_directory_bins):
            os.mkdir(bam_file_cache_directory_bins)
            print(f'crated folder: {bam_file_cache_directory_bins}')
        
        read_1 = f'/scratch/shared/CDM/salterns/reads/paired/{sample_id}_1.fastq'
        read_2 = f'/scratch/shared/CDM/salterns/reads/paired/{sample_id}_2.fastq'
        filename_genomes = f'/scratch/fliu/data/cliff/genomes/ani_library_derep_genomes.txt'
        assert os.path.exists(read_1)
        assert os.path.exists(read_2)
        assert os.path.exists(filename_genomes)
        cmd = [
            'coverm', 'genome', 
            '-1', read_1,
            '-2', read_2,
            '--genome-fasta-list', filename_genomes,
            '-o', f'{bam_file_cache_directory_bins}/coverm.out',
            '--bam-file-cache-directory', bam_file_cache_directory_bins,
            
            '>',  f'{bam_file_cache_directory_bins}/coverm.stdout',
            '2>', f'{bam_file_cache_directory_bins}/coverm.stderr',
        ]
        cmds.append(cmd)

In [96]:
with open('./run_coverm_for_bins.sh', 'w') as fh:
    for cmd in cmds:
        fh.write(' '.join(cmd) + '\n')

### Run CoverM script

run the script in the command line

setup env
```
export PATH=/opt/samtools/:/opt/build/CoverM/target/debug/:/opt/build/strobealign.build/build:$PATH
export LD_LIBRARY_PATH=/opt/htslib/lib
```

run batch
```
cat run_coverm_for_bins.sh | xargs -I CMD --max-procs=10 sh -c CMD
```

In [102]:
import glob
folder = '/scratch/fliu/data/cliff/reads/coverm/'
with open('/home/fliu/cliff_mags/run_samtools_index_for_bins.sh', 'w') as fh:
    for filename in glob.iglob(folder + '**/bins/*.bam', recursive=True):
        filename_index = filename + '.index'
        if not os.path.exists(filename_index):
            cmd = [
                '/opt/samtools/samtools', 'index',
                filename,
                '-o', filename_index
            ]
            fh.write(' '.join(cmd) + '\n')

### Process CoverM results

In [ ]:
import pysam
import pymongo
mongo_client = pymongo.MongoClient('mongodb://sequoia.mcs.anl.gov:27017')
mongo_project = mongo_client['salterns']


def get_reads_mapped(sam):
    reads_mapped = {}
    for read in tqdm(sam.fetch()):
        blocks = read.get_blocks()
        if len(blocks) != 0:
            if read.qname not in reads_mapped:
                reads_mapped[read.qname] = []
            reads_mapped[read.qname].append(read)
    return reads_mapped
sample_id = 'R1_A_D1_MG'
filename_bam = f'/scratch/fliu/data/cliff/reads/coverm/{sample_id}/bins/coverm-genome.{sample_id}_1.fastq.bam'
filename_bam_index = filename_bam + '.index'
c = mongo_project[f'reads_{sample_id}_bins']
with pysam.AlignmentFile(filename_bam, "rb", index_filename=filename_bam_index) as fh_sam:
    for read in tqdm(fh_sam.fetch()):
        doc_read = c.find_one({'_id': read.qname})
        if doc_read is None:
            mongo_project[f'reads_{sample_id}'].insert_one({
                '_id': read.qname,
                'blocks': read.get_blocks(),
                'sequence': read.seq,
                'tags': read.tags,
                'reference_name': read.reference_name,
                'aligned_pairs': read.get_aligned_pairs(),
                'is_paired': read.is_paired,
                'is_mapped': read.is_mapped,
                'is_read1': read.is_read1,
                'is_read2': read.is_read2,
                'is_qcfail': read.is_qcfail,
                'is_reverse': read.is_reverse,
            })
        else:
            assert doc_read['sequence'] == read.seq

5114466it [2:22:18, 604.81it/s]

In [1]:
samples_ids = ['R1_B_D2_MG',
 'R1_B_H2O_MG',
 'R1_C_D2_MG',
 'R2_A_D1_MG',
 'R2_B_D1_MG',
 'R2_B_D2_MG',]

In [ ]:
import pysam
import pymongo
mongo_client = pymongo.MongoClient('mongodb://sequoia.mcs.anl.gov:27017')
mongo_project = mongo_client['salterns']

for sample_id in samples_ids:
    print(sample_id)
    filename_bam = f'/scratch/fliu/data/cliff/reads/coverm/{sample_id}/bins/coverm-genome.{sample_id}_1.fastq.bam'
    filename_bam_index = filename_bam + '.index'
    c = mongo_project[f'reads_{sample_id}_bins']
    with pysam.AlignmentFile(filename_bam, "rb", index_filename=filename_bam_index) as fh_sam:
        reads = []
        for read in fh_sam.fetch():
            doc_read = c.find_one({'_id': read.qname})
            if doc_read is None:
                reads.append({
                        '_id': read.qname,
                        'blocks': read.get_blocks(),
                        'sequence': read.seq,
                        'tags': read.tags,
                        'reference_name': read.reference_name,
                        'aligned_pairs': read.get_aligned_pairs(),
                        'is_paired': read.is_paired,
                        'is_mapped': read.is_mapped,
                        'is_read1': read.is_read1,
                        'is_read2': read.is_read2,
                        'is_qcfail': read.is_qcfail,
                        'is_reverse': read.is_reverse,
                    })
            if len(reads) >= 100:
                c.insert_many(reads)
                reads = []
        if len(reads) >= 0:
            c.insert_many(reads)

R1_B_D2_MG


In [138]:
read.mpos

0

In [105]:
mongo_project

In [4]:
1

1

In [133]:
fh_sam.closed

True

{'_id': 'HISEQ15:203:C7P4WANXX:6:1301:19549:86916_1:N:0:ATTACTCATAGAGG',
 'blocks': [],
 'sequence': 'GGCTGACCCAGCTTCGGGCGGGGCAGAAAATCCGTTTTGCGTCTGTTTCAACGCAGCAAGCTATTGATGCGGCACGCGACTATGCGCAGCAAAAGGCGCAGTACCTCGACCAGATTTCCATTGCGCGGGGCACCTTGGCGCAACGACTGA',
 'tags': [],
 'reference_name': '18a51c005285d036b5bab4187384e8e32b3b42e69d2158efb5c9c223b1866b23~c554005ddae491e80d472a48793a2870b442d8c8eea30e70187b36165670c689',
 'is_paired': True,
 'is_mapped': False,
 'is_read1': True,
 'is_read2': False,
 'is_qcfail': False,
 'is_reverse': False}

In [124]:
read.is_mapped

False

[]

In [114]:
doc_read = c.find_one({'_id': read.qname})

In [48]:
c.find_one_and_delete({})

In [22]:
for sample_id in tqdm(sample_data):
    read_id = sample_data[sample_id].get('Reads')
    assembly_id = sample_data[sample_id].get('Assembly v2 (2018)')
    if not pd.isna(read_id) and not pd.isna(assembly_id):
        read_id = read_id[:-3]
        if read_id not in reads_fastq:
            raise ValueError(f'not found: {read_id}')
        assembly_id = assembly_id + '.filtered.fa'
        if assembly_id not in found_metagenome_assembly:
            raise ValueError(f'not found: {assembly_id}')
        
        

100%|██████████| 38/38 [00:00<00:00, 167772.16it/s]


In [ ]:
def get_start_end(r):
    _min = None
    _max = None
    for a, b in r.get_aligned_pairs():
        if b:
            if _min is None:
                _min = a
            else:
                if a < _min:
                    _min = a
            if _max is None:
                _max = a
            else:
                if a > _max:
                    _max = a
            #print(a, b)
    return _min, _max
def get_contig_sequence2(read):
    file_fasta, contigname = read.reference_name.split('~')
    
    feature = genome_index[file_fasta].features.get_by_id(contigname)
    return feature.seq
def read_align_summary(r, fn_get_contig_sequence):
    contig_sequence = fn_get_contig_sequence(r)
    print(r.get_blocks(), r.tags, r.reference_name)
    _min, _max = get_start_end(r)
    for b0, b1 in r.get_blocks():
        _dist = b1 - b0
        print(b1, b0, _dist)
        print(r.seq, len(r.seq))
        
        print(" " * _min + contig_sequence[b0:b0 + _dist])
def _get_arr_cov(genome, blocks_array):
    
    #for f in genome.features:
        
    for br in progress(blocks_array):
        _genome_id, _contig_id = br[4].split('~')
        arr_cov = contig_cov[_contig_id]
        for b0, b1 in br[1]:
            _dist = b1 - b0
            for i in range(_dist):
                arr_cov[b0 + i] += 1
    return contig_cov
def get_contig_sequence(r):
    contig_name = r.reference_name.split('~')[1]
    _hash = contig_id_to_h[contig_name]
    if len(_hash) == 1:
        return hash_to_seq[list(_hash)[0]]
    raise ValueError('!')
#contig_sequence = get_contig_sequence(r.reference_name)
#len(contig_sequence)

In [ ]:
1